In [1]:
import os
import cv2
import numpy as np
import matplotlib.pyplot as plt

import sys
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "../..")))

from src.data_postprocessing import obtain_shoreline
from src.data_processing.dataset_loader import CoastData
from scipy.spatial import cKDTree


In [2]:
def compute_distance(coords_pred, coords_gt):
    # Distance from predicted to GT
    # Create a KDTree for the Ground Truth coordinates
    tree_gt = cKDTree(coords_gt)

    # Find the nearest neighbors in the Ground Truth for each coordinate in the predicted mask
    dists_pred_to_gt, _ = tree_gt.query(coords_pred)

    # Distance from GT to predicted
    # Create a KDTree for the predicted coordinates
    tree_pred = cKDTree(coords_pred)
    # Find the nearest neighbors in the predicted for each coordinate in the GT mask
    dists_gt_to_pred, _ = tree_pred.query(coords_gt)

    return dists_pred_to_gt, dists_gt_to_pred

In [3]:
def calculate_dataset(data_path, in_meters=False, stations=[]):
    data = CoastData(data_path)

    if 'global' not in stations:
        stations.append('global')

    filtered_data = data.get_images(get_all_metadata=True, get_mask=False) # All the data

    global_distance_all_points = {}
    
    for item in filtered_data:
        coords_gt_u = item['metadata']["image"]["shoreline"]["coordinates"]['u']
        coords_gt_v = item['metadata']["image"]["shoreline"]["coordinates"]['v']
        coords_gt = np.column_stack((coords_gt_u, coords_gt_v))

        coords_pred_u = item['metadata']['image']['predicted_shoreline']["coordinates"]['u']
        coords_pred_v = item['metadata']['image']['predicted_shoreline']["coordinates"]['v']
        coords_pred = np.column_stack((coords_pred_u, coords_pred_v))

        dists_pred_to_gt, dists_gt_to_pred = compute_distance(coords_pred, coords_gt)

        station_name = item['metadata']['image']['site']['CSname'] if len(stations) > 1 else 'global'

        new_stations = [station_name, 'global'] if len(stations) > 1 else ['global']

        for station in new_stations:
            if station not in global_distance_all_points:
                global_distance_all_points[station] = {
                    "dist_pred_to_gt": [],
                    "dist_gt_to_pred": [],
                    "pred_points": 0,
                    "gt_points": 0
                }
            global_distance_all_points[station]["dist_pred_to_gt"].extend(dists_pred_to_gt)
            global_distance_all_points[station]["dist_gt_to_pred"].extend(dists_gt_to_pred)
            global_distance_all_points[station]["pred_points"] += len(coords_pred)
            global_distance_all_points[station]["gt_points"] += len(coords_gt)

    pixel_scale = 0.5 if in_meters else 1

    for station in stations:
        global_distance_all_points[station]["dist_pred_to_gt"] = np.array(global_distance_all_points[station]["dist_pred_to_gt"]) * pixel_scale
        global_distance_all_points[station]["dist_gt_to_pred"] = np.array(global_distance_all_points[station]["dist_gt_to_pred"]) * pixel_scale

        mean_dist_pred_to_gt = np.mean(global_distance_all_points[station]["dist_pred_to_gt"])
        std_dist_pred_to_gt = np.std(global_distance_all_points[station]["dist_pred_to_gt"])
        rmsd_dist_pred_to_gt = np.sqrt(np.mean(np.square(global_distance_all_points[station]["dist_pred_to_gt"])))
        mean_dist_gt_to_pred = np.mean(global_distance_all_points[station]["dist_gt_to_pred"])
        std_dist_gt_to_pred = np.std(global_distance_all_points[station]["dist_gt_to_pred"])
        rmsd_dist_gt_to_pred = np.sqrt(np.mean(np.square(global_distance_all_points[station]["dist_gt_to_pred"])))
        q3_dist_pred_to_gt = np.percentile(global_distance_all_points[station]["dist_pred_to_gt"], 75)
        q3_dist_gt_to_pred = np.percentile(global_distance_all_points[station]["dist_gt_to_pred"], 75)

        ratio = global_distance_all_points[station]["pred_points"] / global_distance_all_points[station]["gt_points"]

        print(f"Station: {station}")
        print(f"Number of points pred: {len(global_distance_all_points[station]['dist_pred_to_gt'])}, Number of points GT: {len(global_distance_all_points[station]['dist_gt_to_pred'])}, Ratio: {ratio:.4f}")
        print(f"Mean Absolute Distance (pred->Gt) global: {np.mean(mean_dist_pred_to_gt):.4f} ({std_dist_pred_to_gt:.4f})")
        print(f"Mean Absolute Distance (Gt->pred) global: {np.mean(mean_dist_gt_to_pred):.4f} ({std_dist_gt_to_pred:.4f})")
        print(f"RMSD (pred->Gt) global: {np.mean(rmsd_dist_pred_to_gt):.4f}")
        print(f"RMSD (Gt->pred) global: {np.mean(rmsd_dist_gt_to_pred):.4f}")
        print(f"Q3 75th Percentile (pred->Gt) global: {np.mean(q3_dist_pred_to_gt):.4f}")
        print(f"Q3 75th Percentile (Gt->pred) global: {np.mean(q3_dist_gt_to_pred):.4f}\n")



In [4]:
# stations = ['agrelo', 'arenaldentem', 'cadiz', 'cies', 'samarador']
stations = ['global']

## Experiment 2

### Rectified

In [5]:
data_path_unet = os.path.abspath(os.path.join(os.getcwd(), "../../outputs/experiment2/rectified/UNet/"))
data_path_attention_unet = os.path.abspath(os.path.join(os.getcwd(), "../../outputs/experiment2/rectified/AttentionUNet/"))
data_path_deeplabv3 = os.path.abspath(os.path.join(os.getcwd(), "../../outputs/experiment2/rectified/DeepLabV3/"))
data_path_ducknet = os.path.abspath(os.path.join(os.getcwd(), "../../outputs/experiment2/rectified/DuckNet/"))

print(f"UNet rectified results:")
calculate_dataset(data_path_unet, stations=stations, in_meters=True)
print(f"Attention UNet results:")
calculate_dataset(data_path_attention_unet, stations=stations, in_meters=True)
print(f"DeepLabV3 rectified results:")
calculate_dataset(data_path_deeplabv3, stations=stations, in_meters=True)
print(f"DuckNet results:")
calculate_dataset(data_path_ducknet, stations=stations, in_meters=True)

UNet rectified results:
CoastData: global - 174 images
Station: global
Number of points pred: 102311, Number of points GT: 88650, Ratio: 1.1541
Mean Absolute Distance (pred->Gt) global: 6.1250 (9.2623)
Mean Absolute Distance (Gt->pred) global: 4.9095 (10.3362)
RMSD (pred->Gt) global: 11.1043
RMSD (Gt->pred) global: 11.4429
Q3 75th Percentile (pred->Gt) global: 6.9462
Q3 75th Percentile (Gt->pred) global: 5.3151

Attention UNet results:
CoastData: global - 174 images
Station: global
Number of points pred: 103315, Number of points GT: 88650, Ratio: 1.1654
Mean Absolute Distance (pred->Gt) global: 6.6450 (10.9385)
Mean Absolute Distance (Gt->pred) global: 5.5810 (12.9913)
RMSD (pred->Gt) global: 12.7987
RMSD (Gt->pred) global: 14.1394
Q3 75th Percentile (pred->Gt) global: 6.8007
Q3 75th Percentile (Gt->pred) global: 5.3852

DeepLabV3 rectified results:
CoastData: global - 174 images
Station: global
Number of points pred: 97031, Number of points GT: 88650, Ratio: 1.0945
Mean Absolute Dista

### Oblique

In [6]:
# Oblique results
data_path_unet = os.path.abspath(os.path.join(os.getcwd(), "../../outputs/experiment2/oblique/UNet/"))
data_path_attention_unet = os.path.abspath(os.path.join(os.getcwd(), "../../outputs/experiment2/oblique/AttentionUNet/"))
data_path_deeplabv3 = os.path.abspath(os.path.join(os.getcwd(), "../../outputs/experiment2/oblique/DeepLabV3/"))
data_path_ducknet = os.path.abspath(os.path.join(os.getcwd(), "../../outputs/experiment2/oblique/DuckNet/"))

print("UNet results:")
calculate_dataset(data_path_unet, stations=stations)
print("Attention UNet results:")
calculate_dataset(data_path_attention_unet, stations=stations)
print("DeepLabV3 results:")
calculate_dataset(data_path_deeplabv3, stations=stations)
print("DuckNet results:")
calculate_dataset(data_path_ducknet, stations=stations)

UNet results:
CoastData: global - 174 images
Station: global
Number of points pred: 406665, Number of points GT: 341638, Ratio: 1.1903
Mean Absolute Distance (pred->Gt) global: 59.4567 (192.9484)
Mean Absolute Distance (Gt->pred) global: 62.7271 (239.6138)
RMSD (pred->Gt) global: 201.9015
RMSD (Gt->pred) global: 247.6883
Q3 75th Percentile (pred->Gt) global: 26.9258
Q3 75th Percentile (Gt->pred) global: 18.6815

Attention UNet results:
CoastData: global - 174 images
Station: global
Number of points pred: 435346, Number of points GT: 341638, Ratio: 1.2743
Mean Absolute Distance (pred->Gt) global: 152.3757 (382.4166)
Mean Absolute Distance (Gt->pred) global: 119.6105 (385.6378)
RMSD (pred->Gt) global: 411.6562
RMSD (Gt->pred) global: 403.7613
Q3 75th Percentile (pred->Gt) global: 50.4480
Q3 75th Percentile (Gt->pred) global: 26.0000

DeepLabV3 results:
CoastData: global - 174 images
Station: global
Number of points pred: 353246, Number of points GT: 341638, Ratio: 1.0340
Mean Absolute Di

## Experiment 3

### Rectified

In [7]:
data_path_deeplabv3_256x256 = os.path.abspath(os.path.join(os.getcwd(), "../../outputs/experiment3/rectified/DeepLabV3_256x256/"))
data_path_deeplabv3_256x512 = os.path.abspath(os.path.join(os.getcwd(), "../../outputs/experiment3/rectified/DeepLabV3_256x512/"))
data_path_deeplabv3_256x1024 = os.path.abspath(os.path.join(os.getcwd(), "../../outputs/experiment3/rectified/DeepLabV3_256x1024/"))
data_path_deeplabv3_512x512 = os.path.abspath(os.path.join(os.getcwd(), "../../outputs/experiment3/rectified/DeepLabV3_512x512/"))

print(f"DeepLabV3 rectified 256x256 results:")
calculate_dataset(data_path_deeplabv3_256x256, stations=stations, in_meters=True)
print(f"DeepLabV3 rectified 256x512 results:")
calculate_dataset(data_path_deeplabv3_256x512, stations=stations, in_meters=True)
print(f"DeepLabV3 rectified 256x1024 results:")
calculate_dataset(data_path_deeplabv3_256x1024, stations=stations, in_meters=True)
print(f"DeepLabV3 rectified 512x512 results:")
calculate_dataset(data_path_deeplabv3_512x512, stations=stations, in_meters=True)

DeepLabV3 rectified 256x256 results:
CoastData: global - 174 images
Station: global
Number of points pred: 97031, Number of points GT: 88650, Ratio: 1.0945
Mean Absolute Distance (pred->Gt) global: 5.0748 (6.9507)
Mean Absolute Distance (Gt->pred) global: 4.1832 (5.6541)
RMSD (pred->Gt) global: 8.6061
RMSD (Gt->pred) global: 7.0334
Q3 75th Percentile (pred->Gt) global: 6.0415
Q3 75th Percentile (Gt->pred) global: 5.0000

DeepLabV3 rectified 256x512 results:
CoastData: global - 174 images
Station: global
Number of points pred: 93561, Number of points GT: 88650, Ratio: 1.0554
Mean Absolute Distance (pred->Gt) global: 4.6650 (6.6048)
Mean Absolute Distance (Gt->pred) global: 3.9589 (5.1524)
RMSD (pred->Gt) global: 8.0861
RMSD (Gt->pred) global: 6.4977
Q3 75th Percentile (pred->Gt) global: 5.3151
Q3 75th Percentile (Gt->pred) global: 4.7170

DeepLabV3 rectified 256x1024 results:
CoastData: global - 174 images
Station: global
Number of points pred: 92530, Number of points GT: 88650, Ratio: 